# OKR paper (arXiv 2607.00185): Validations, Sensitivity Matrix & B7 Audit (v2)

**Run all cells top to bottom (local or Kaggle with Internet ON).** Edit only the `CONFIG` cells.

- Supports three reproducible modes via `RUN_MODE`:
  - `'v1-exact'` — exact 27 features from `01_pipeline_khipu.ipynb`, `n = 619` (includes the 7 empty Dallas Museum of Art records `KH0601–KH0607`)
  - `'v1-corregida'` — exact 27 features from v1, `n = 612` (excludes the 7 records with zero cords confirmed by OKR)
  - `'robustez'` — 23 reconstructed features from `cord`, `n = 612`
  - `'matrix'` — runs all three modes and outputs a unified sensitivity table (`results/table_sensitivity_matrix.csv`), plus full detailed diagnostics and figures for `PRIMARY_MODE`
- Strict role-based column mapping with `assert` guards (no silent column skipping) and `>= 95%` metadata/documentation join coverage checks.
- Includes **A1–A3** (mandatory audits), **B1–B5** (size control, null models, cataloger/documentation confound, grouped CV, imperial ablations), **B6** (knot-type bigram TF-IDF on the exact same feature set), and **B7** (`CREATED_BY = jclindaniel` audit, reproducible exclusion artifact, `612 / 467 / 430` progression, 18-config hyperparameter sweep, and `1,000 + 1,000` null models).


In [1]:
import os, sys, subprocess, importlib
# Must be set BEFORE importing numpy/scipy/umap/numba to prevent thread oversubscription
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(_v, '1')

def _need(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
for p, m in [('umap-learn', 'umap'), ('hdbscan', None), ('xgboost', None), ('shap', None), ('requests', None), ('scipy', None)]:
    _need(p, m)

import re, json, zipfile, sqlite3, warnings, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import umap
import hdbscan
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import entropy as scipy_entropy, chi2_contingency
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, adjusted_rand_score, f1_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure repo root is on sys.path so `from src.b7_nulls import ...` works from notebooks/ or repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.b7_nulls import run_null_suite, format_p

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print({k: getattr(__import__(k), '__version__', 'n/a') for k in ['numpy', 'pandas', 'scipy', 'umap', 'hdbscan', 'xgboost', 'shap', 'sklearn']})

STATUS = {}
class step:
    def __init__(self, name): self.name = name
    def __enter__(self): print(f'\n===== {self.name} ====='); return self
    def __exit__(self, et, ev, tb):
        STATUS[self.name] = 'OK' if et is None else f'FAILED: {et.__name__}: {str(ev)[:180]}'
        if et is not None:
            print('!! step failed ->', STATUS[self.name])
            traceback.print_exception(et, ev, tb, limit=3)
        return et is not None


C:\Users\maria\.gemini\antigravity\brain\fc73bd56-8b09-40d2-8033-004b16e8895b\scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'numpy': '2.5.5', 'pandas': '3.1.1', 'scipy': '1.18.1', 'umap': '0.6.0', 'hdbscan': 'n/a', 'xgboost': '3.4.1', 'shap': '0.54.1', 'sklearn': '1.9.1'}


In [2]:
# ===================== CONFIG: edit only this cell =====================
RUN_MODE       = 'matrix'    # 'matrix' | 'v1-exact' | 'v1-corregida' | 'robustez'
PRIMARY_MODE   = 'v1-exact'  # detailed tables/figures/exports when RUN_MODE == 'matrix'

FEATURES_CSV   = None        # Optional CSV path (used only when RUN_MODE != 'matrix'); None = build from OKR SQLite
DB_PATH        = None        # Path to khipu.db, or None -> auto-detect local data/ or download from Zenodo
ZENODO_ID      = 18025748    # OKR record (DOI 10.5281/zenodo.18025748)
OKR_VERSION    = '2.1.0'     # Zenodo release tag

RESULTS_DIR    = str(REPO_ROOT / 'results')  # Versioned small CSV tables
OUTPUT_DIR     = str(REPO_ROOT / 'outputs')  # Gitignored figures & full JSON

ID_COL         = None        # None -> auto-detect (khipu_id)
REGION_COL     = None        # None -> auto-detect
MUSEUM_COL     = None        # None -> auto-detect
NAME_COL       = None        # Column with catalog names like 'UR126'/'AS070'/'QU01'
IMPERIAL_LABEL = None        # None -> auto (label containing 'inka' & 'late')
COLL_CLUSTER   = None        # None -> auto (2nd largest non-noise cluster)

UMAP_KW        = dict(n_neighbors=15, min_dist=0.1, metric='euclidean')
HDB_KW         = dict(min_cluster_size=10, min_samples=5, metric='euclidean')
N_SIM          = 100         # Null-model simulations per null in sensitivity matrix
N_PERM         = 100         # Permutations for imperial-class tests (B5)
RUN_NULLS      = False       # Set True to recompute B2 in matrix (loads results/table_sensitivity_matrix.csv if False and present)
RUN_PERMS      = False       # Set True to recompute B5 N_PERM=100 permutations (~35 min across 3 modes); False still computes exact observed F1s
# ========================================================================
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Data Loading, Feature Builders & Role-Based Column Mapping (with `assert` guards)


In [3]:
ROLE_SCHEMAS = {
    'v1': {
        'twist_ratio':   ['ratio_SZ'],
        'twist_and_dir': ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ'],
        'geometry':      ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns'],
        'size':          ['n_cords', 'n_knots'],
    },
    'v2': {
        'twist_ratio':   ['twist_ratio'],
        'twist_and_dir': ['n_twist_S', 'n_twist_Z', 'twist_ratio', 'knot_dir_S', 'knot_dir_Z'],
        'geometry':      ['n_cords', 'cord_len_mean', 'cord_len_std', 'max_level', 'subsidiary_ratio', 'n_knots', 'knot_turns_mean'],
        'size':          ['n_cords', 'n_knots'],
    }
}
EXPECTED_COUNTS = {
    'v1': {'twist_ratio': 1, 'twist_and_dir': 6, 'geometry': 7, 'size': 2},
    'v2': {'twist_ratio': 1, 'twist_and_dir': 5, 'geometry': 7, 'size': 2},
}

def resolve_roles(columns, verbose=True):
    """Detect schema (v1 or v2), map columns by role, print assignment, and assert completeness."""
    cols = set(columns)
    scores = {sname: len(cols & set().union(*rdict.values())) for sname, rdict in ROLE_SCHEMAS.items()}
    schema = max(scores, key=scores.get)
    rdict = ROLE_SCHEMAS[schema]
    assigned = {}
    for role, expected in rdict.items():
        present = [c for c in expected if c in cols]
        missing = [c for c in expected if c not in cols]
        exp_n = EXPECTED_COUNTS[schema][role]
        assert len(present) == exp_n and not missing, (
            f"Incomplete role '{role}' under schema '{schema}': "
            f"expected {exp_n} cols {expected}, found {len(present)} ({present}), missing {missing}"
        )
        assigned[role] = present
    if verbose:
        print(f"[resolve_roles] Detected schema: {schema}")
        for role, clist in assigned.items():
            print(f"  {role:15s} ({len(clist)}): {clist}")
    return schema, assigned

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

def open_db(db_path=None):
    path = db_path or DB_PATH
    if not path:
        for root in [str(REPO_ROOT / 'data'), '.', '..', 'data', 'okr_data']:
            if os.path.isdir(root):
                for d, _, fs in os.walk(root):
                    for f in fs:
                        if f.lower().endswith(('.sqlite', '.sqlite3', '.db')) and 'khipu' in f.lower():
                            path = os.path.join(d, f)
                            break
                    if path: break
            if path: break
    if not path:
        rec = requests.get(f'https://zenodo.org/api/records/{ZENODO_ID}', timeout=60).json()
        files = rec.get('files', [])
        cand = []
        for f in files:
            name = f.get('key') or f.get('filename') or ''
            url = (f.get('links') or {}).get('self') or (f.get('links') or {}).get('download')
            if name.lower().endswith(('.sqlite', '.sqlite3', '.db', '.zip')):
                cand.append((name, url))
        assert cand, 'No sqlite/zip in the Zenodo record: download manually and set DB_PATH'
        cand.sort(key=lambda t: not t[0].lower().endswith(('.sqlite', '.sqlite3', '.db')))
        name, url = cand[0]
        os.makedirs('okr_data', exist_ok=True)
        dst = os.path.join('okr_data', name)
        if not os.path.exists(dst):
            r = requests.get(url, timeout=600); r.raise_for_status(); open(dst, 'wb').write(r.content)
        if dst.lower().endswith('.zip'):
            zipfile.ZipFile(dst).extractall('okr_data')
            dbs = [os.path.join(d, f) for d, _, fs in os.walk('okr_data') for f in fs if f.lower().endswith(('.sqlite', '.sqlite3', '.db'))]
            assert dbs, 'zip has no sqlite'
            dst = dbs[0]
        path = dst
    print('DB:', path)
    return sqlite3.connect(path)

def q(con, table):
    names = {n.lower(): n for (n,) in con.execute("select name from sqlite_master where type in ('table','view')")}
    t = names.get(table.lower())
    assert t, f'table {table} not found; have {list(names.values())}'
    df = pd.read_sql(f'select * from "{t}"', con)
    df.columns = [c.lower() for c in df.columns]
    return df

def _doc_features_from_db(con, kh_index):
    cord = q(con, 'cord')
    knot = q(con, 'knot')
    c_k = pick(cord.columns, ['khipu_id']); c_id = pick(cord.columns, ['cord_id'])
    c_tw = pick(cord.columns, ['twist', 'twist_code', 'cord_twist'])
    c_att = pick(cord.columns, ['attachment_type'])
    k_c = pick(knot.columns, ['cord_id']); k_di = pick(knot.columns, ['direction', 'knot_direction', 'dir'])
    if pick(knot.columns, ['khipu_id']) is None:
        knot = knot.merge(cord[[c_id, c_k]], left_on=k_c, right_on=c_id, how='inner', suffixes=('', '_c'))
        k_k = c_k if c_k in knot.columns else c_k + '_c'
    else:
        k_k = pick(knot.columns, ['khipu_id'])
    tw = cord[c_tw].astype(str).str.upper().str.strip() if c_tw else pd.Series('U', index=cord.index)
    kd = knot[k_di].astype(str).str.upper().str.strip() if k_di else pd.Series('U', index=knot.index)
    D = pd.DataFrame(index=cord[c_k].unique())
    D['pct_twist_U'] = (~tw.isin(['S', 'Z'])).groupby(cord[c_k]).mean()
    D['pct_knot_dir_U'] = (~kd.isin(['S', 'Z'])).groupby(knot[k_k]).mean()
    if c_att:
        D['pct_side_U'] = (~cord[c_att].astype(str).str.upper().isin(['R', 'V'])).groupby(cord[c_k]).mean()
    D.index = D.index.astype(str)
    D.index.name = 'khipu_id'
    return D

def build_v1_from_db(con, drop_empty=False):
    """Exact replication of 01_pipeline_khipu.ipynb (27 features)."""
    kh = q(con, 'khipu_main')
    cord = q(con, 'cord')
    knot = q(con, 'knot')
    colors = q(con, 'ascher_cord_color')

    cord_feats = cord.groupby('khipu_id').agg(
        n_cords          = ('cord_id', 'count'),
        mean_length      = ('cord_length', 'mean'),
        std_length       = ('cord_length', 'std'),
        max_cord_level   = ('cord_level', 'max'),
        n_subsidiary     = ('cord_level', lambda x: (x > 1).sum()),
        ratio_subsidiary = ('cord_level', lambda x: (x > 1).mean()),
        twist_S          = ('twist', lambda x: (x == 'S').sum()),
        twist_Z          = ('twist', lambda x: (x == 'Z').sum()),
    ).reset_index()
    cord_feats['ratio_SZ'] = cord_feats['twist_S'] / (cord_feats['twist_S'] + cord_feats['twist_Z'] + 1e-9)
    cord_feats['std_length'] = cord_feats['std_length'].fillna(0)

    merged = knot.merge(cord[['cord_id', 'khipu_id']], on='cord_id', how='left')
    type_dummies = pd.get_dummies(merged['type_code'], prefix='knot')
    knot_type_feats = pd.concat([merged[['khipu_id']], type_dummies], axis=1).groupby('khipu_id').sum().reset_index()
    knot_general = merged.groupby('khipu_id').agg(
        n_knots    = ('type_code', 'count'),
        mean_turns = ('num_turns', 'mean'),
        knot_dir_S = ('direction', lambda x: (x == 'S').sum()),
        knot_dir_Z = ('direction', lambda x: (x == 'Z').sum()),
    ).reset_index()
    knot_general['ratio_knot_SZ'] = knot_general['knot_dir_S'] / (knot_general['knot_dir_S'] + knot_general['knot_dir_Z'] + 1e-9)
    knot_feats = knot_general.merge(knot_type_feats, on='khipu_id', how='left')

    def _color_ent(s):
        c = s.dropna().value_counts(normalize=True)
        return float(scipy_entropy(c)) if len(c) > 1 else 0.0
    color_feats = colors.groupby('khipu_id').agg(
        n_unique_colors = ('color_cd_1', 'nunique'),
        color_entropy   = ('color_cd_1', _color_ent),
        has_multicolor  = ('color_cd_2', lambda x: int(x.notna().any())),
    ).reset_index()

    fm = kh[['khipu_id']].copy()
    fm = fm.merge(cord_feats, on='khipu_id', how='left')
    fm = fm.merge(knot_feats, on='khipu_id', how='left')
    fm = fm.merge(color_feats, on='khipu_id', how='left')
    fm = fm.set_index('khipu_id')
    fm.index = fm.index.astype(str)
    fm.index.name = 'khipu_id'
    fm = fm.fillna(0)
    if drop_empty:
        fm = fm[fm['n_cords'] > 0].copy()
    D = _doc_features_from_db(con, fm.index)
    return fm, D, kh

def build_v2_from_db(con):
    """Original v2 reconstructed 23-feature matrix (starting from cord, n=612)."""
    kh, cord, knot = q(con, 'khipu_main'), q(con, 'cord'), q(con, 'knot')
    c_k = pick(cord.columns, ['khipu_id']); c_id = pick(cord.columns, ['cord_id'])
    c_len = pick(cord.columns, ['cord_length', 'length']); c_tw = pick(cord.columns, ['twist', 'twist_code', 'cord_twist'])
    c_lv = pick(cord.columns, ['cord_level', 'level', 'hierarchical_level']); c_att = pick(cord.columns, ['attachment_type'])
    k_c = pick(knot.columns, ['cord_id']); k_ty = pick(knot.columns, ['type_code', 'knot_type', 'type'])
    k_di = pick(knot.columns, ['direction', 'knot_direction', 'dir']); k_tu = pick(knot.columns, ['num_turns', 'turns'])
    if pick(knot.columns, ['khipu_id']) is None:
        knot = knot.merge(cord[[c_id, c_k]], left_on=k_c, right_on=c_id, how='inner', suffixes=('', '_c'))
        k_k = c_k if c_k in knot.columns else c_k + '_c'
    else:
        k_k = pick(knot.columns, ['khipu_id'])
    g = cord.groupby(c_k)
    F = pd.DataFrame(index=g.size().index)
    F['n_cords'] = g.size()
    F['cord_len_mean'] = g[c_len].mean(); F['cord_len_std'] = g[c_len].std()
    lv = pd.to_numeric(cord[c_lv], errors='coerce')
    F['max_level'] = lv.groupby(cord[c_k]).max()
    F['subsidiary_ratio'] = (lv > 1).groupby(cord[c_k]).mean()
    tw = cord[c_tw].astype(str).str.upper().str.strip()
    F['n_twist_S'] = (tw == 'S').groupby(cord[c_k]).sum(); F['n_twist_Z'] = (tw == 'Z').groupby(cord[c_k]).sum()
    F['twist_ratio'] = F['n_twist_S'] / (F['n_twist_S'] + F['n_twist_Z']).replace(0, np.nan)
    gk = knot.groupby(k_k)
    F['n_knots'] = gk.size()
    F['knot_turns_mean'] = gk[k_tu].apply(lambda s: pd.to_numeric(s, errors='coerce').mean())
    kd = knot[k_di].astype(str).str.upper().str.strip()
    F['knot_dir_S'] = (kd == 'S').groupby(knot[k_k]).sum(); F['knot_dir_Z'] = (kd == 'Z').groupby(knot[k_k]).sum()
    ct = pd.crosstab(knot[k_k], knot[k_ty].astype(str))
    ct = ct[ct.sum().sort_values(ascending=False).index[:8]]
    ct.columns = [f'knot_type_{c}' for c in ct.columns]
    F = F.join(ct)
    col = q(con, 'ascher_cord_color')
    cc = pick(col.columns, ['cord_id']); cl = [c for c in col.columns if 'color' in c and c != cc]
    col = col.merge(cord[[c_id, c_k]], left_on=cc, right_on=c_id, how='inner', suffixes=('', '_c'))
    kk = c_k if c_k in col.columns else c_k + '_c'
    def _ent(s):
        p = s.value_counts(normalize=True).values
        return float(-(p * np.log(p)).sum())
    F['n_colors'] = col.groupby(kk)[cl[0]].nunique()
    F['color_entropy'] = col.groupby(kk)[cl[0]].apply(_ent)
    F['multicolor'] = (F['n_colors'] > 1).astype(int)
    F.index = F.index.astype(str); F.index.name = 'khipu_id'
    D = _doc_features_from_db(con, F.index)
    return F.fillna(0), D, kh

def load_dataset(mode, con, features_csv=None, verbose=True):
    """Load F, DOC, META, roles, and enforce >=95% join coverage asserts."""
    if features_csv is not None:
        raw = pd.read_csv(features_csv)
        id_c = ID_COL or pick(raw.columns, ['khipu_id', 'id', 'okr_id', 'khipu_name', 'name']) or raw.columns[0]
        raw = raw.set_index(id_c); raw.index = raw.index.astype(str); raw.index.name = 'khipu_id'
        raw = raw[[c for c in raw.columns if not re.match(r'^unnamed', c, re.I)]]
        csv_cl = [c for c in raw.columns if re.search(r'cluster|hdbscan|umap|^label', c, re.I)]
        D_csv = raw[[c for c in raw.columns if re.search(r'(^pct_.*_u$|unrecorded|pct_.*unknown)', c, re.I)]].copy()
        META = raw.select_dtypes(exclude=[np.number]).copy()
        F = raw.select_dtypes(include=[np.number]).drop(columns=list(D_csv.columns) + csv_cl, errors='ignore')
        kh = q(con, 'khipu_main')
        D = D_csv if D_csv.shape[1] > 0 else _doc_features_from_db(con, F.index)
    elif mode == 'v1-exact':
        F, D, kh = build_v1_from_db(con, drop_empty=False)
        META = pd.DataFrame(index=F.index)
    elif mode == 'v1-corregida':
        F, D, kh = build_v1_from_db(con, drop_empty=True)
        META = pd.DataFrame(index=F.index)
    elif mode == 'robustez':
        F, D, kh = build_v2_from_db(con)
        META = pd.DataFrame(index=F.index)
    else:
        raise ValueError(f'Unknown mode: {mode}')

    idx_s = set(map(str, F.index))
    best_key = max(kh.columns, key=lambda c: len(idx_s & set(kh[c].astype(str))))
    kh_idx = kh.drop_duplicates(best_key).set_index(best_key)
    kh_idx.index = kh_idx.index.astype(str)
    meta_overlap = len(idx_s & set(kh_idx.index)) / len(F)
    assert meta_overlap >= 0.95, f"Metadata join coverage too low: {meta_overlap:.1%} (<95%) using key '{best_key}'"

    META = META.join(kh_idx[[c for c in kh_idx.columns if c not in META.columns]], how='left')
    DOC_raw = D.reindex(F.index)
    doc_coverage = float(DOC_raw.notna().any(axis=1).mean())
    assert doc_coverage >= 0.95, f"Documentation join coverage too low: {doc_coverage:.1%} (<95%)"

    DOC = DOC_raw.fillna(1.0)
    schema, roles = resolve_roles(F.columns, verbose=verbose)
    if verbose:
        print(f"[{mode}] n={len(F)}, p={F.shape[1]}, meta_cov={meta_overlap:.1%}, doc_cov={doc_coverage:.1%}")
    return F, DOC, META, schema, roles


## 2. Full Validation Engine (Baseline, A1–A3, B1–B6)


In [4]:
def run_cluster(Z, seed=SEED):
    emb = umap.UMAP(n_components=2, random_state=seed, **UMAP_KW).fit_transform(Z)
    lab = hdbscan.HDBSCAN(**HDB_KW).fit_predict(emb)
    ok = lab >= 0
    sil = silhouette_score(emb[ok], lab[ok]) if ok.sum() > 1 and len(set(lab[ok])) > 1 else np.nan
    return emb, lab, sil

def cramers_v(ct):
    chi2 = chi2_contingency(ct)[0]; n_tot = ct.values.sum(); k_min = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n_tot * k_min))) if k_min > 0 else np.nan

def build_bigram_df(con):
    """Section 10 of v1: TF-IDF over knot-type bigrams ordered by KHIPU_ID, CORD_ID, KNOT_ORDINAL."""
    knot_seq = pd.read_sql('''
        SELECT k.CORD_ID, c.KHIPU_ID, k.TYPE_CODE, k.KNOT_ORDINAL
        FROM knot k
        JOIN cord c ON k.CORD_ID = c.CORD_ID
        WHERE k.TYPE_CODE IS NOT NULL AND k.KNOT_ORDINAL IS NOT NULL
        ORDER BY c.KHIPU_ID, k.CORD_ID, k.KNOT_ORDINAL
    ''', con)
    seqs = knot_seq.groupby('KHIPU_ID')['TYPE_CODE'].apply(list).reset_index()
    seqs['bigram_text'] = seqs['TYPE_CODE'].apply(
        lambda s: ' '.join(f'{s[i]}-{s[i+1]}' for i in range(len(s) - 1))
    )
    tfidf = TfidfVectorizer(max_features=50, min_df=5)
    X_ng = tfidf.fit_transform(seqs['bigram_text']).toarray()
    ng_cols = [f'ngram_{f}' for f in tfidf.get_feature_names_out()]
    ng_df = pd.DataFrame(X_ng, columns=ng_cols, index=seqs['KHIPU_ID'].astype(str))
    ng_df.index.name = 'khipu_id'
    return ng_df

def run_full_analysis(mode, con, features_csv=None, run_nulls=RUN_NULLS, n_sim=N_SIM, n_perm=N_PERM, verbose=True):
    R = {'mode': mode}
    F, DOC, META, schema, roles = load_dataset(mode, con, features_csv=features_csv, verbose=verbose)
    R['n'] = int(len(F)); R['n_features'] = int(F.shape[1]); R['schema'] = schema

    REGION = REGION_COL or pick(META.columns, ['region', 'geographic_region', 'provenance_region', 'provenance'])
    MUSEUM = MUSEUM_COL or pick(META.columns, ['museum_name', 'museum', 'museum_collection'])
    NAME   = NAME_COL   or pick(META.columns, ['investigator_num', 'khipu_name', 'orig_khipu_name', 'kfg_name', 'name', 'nickname'])
    CATALOGER = META[NAME].astype(str).str.extract(r'^([A-Za-z]{2})')[0].str.upper() if NAME else pd.Series(np.nan, index=F.index)

    # ---- Baseline clustering ----
    Z0 = StandardScaler().fit_transform(F.values)
    emb0, lab0, sil0 = run_cluster(Z0)
    sizes = pd.Series(lab0).value_counts().sort_index()
    non_noise_sizes = sorted([int(v) for k, v in sizes.items() if k >= 0])
    R['baseline'] = dict(
        n_clusters=int((sizes.index >= 0).sum()),
        silhouette=float(sil0),
        sizes={int(k): int(v) for k, v in sizes.items()},
        sorted_sizes=non_noise_sizes,
    )
    if verbose:
        print(f"[{mode}] Baseline clusters: {sizes.to_dict()} | silhouette={sil0:.3f}")
    if mode == 'v1-exact':
        ref_sizes = [17, 160, 442]
        sizes_ok = (len(non_noise_sizes) == 3 and all(abs(a - b) <= 5 for a, b in zip(non_noise_sizes, ref_sizes)))
        sil_ok = (abs(sil0 - 0.769) <= 0.08)
        if not (sizes_ok and sil_ok):
            print(f"WARNING [v1-exact]: differs from v1 paper (expected ~{ref_sizes}, sil~0.769; got {non_noise_sizes}, sil={sil0:.3f})")
        else:
            print(f"[v1-exact] Reproduces v1 within UMAP tolerance (got {non_noise_sizes}, sil={sil0:.3f} vs ref {ref_sizes}, 0.769)")

    CL = pd.Series(lab0, index=F.index, name='cluster')
    big = [c for c in sizes.index if c >= 0]
    order = sizes.loc[big].sort_values(ascending=False).index.tolist()
    MAIN_CLUSTER = order[0]
    COLL_CL = COLL_CLUSTER if COLL_CLUSTER is not None else (order[1] if len(order) > 1 else order[0])

    # ---- A1: Label audit (pandas 2.x & 3.x safe) ----
    reg = META[REGION].fillna('').astype(str).str.strip() if REGION else pd.Series('', index=F.index)
    bad = {'', 'nan', 'none', 'unknown', 'na', 'n/a', 'null', '<na>'}
    nonnull = reg[~reg.str.lower().isin(bad)]
    vc = nonnull.value_counts()
    vc10 = nonnull.where(nonnull.map(vc) >= 10, 'Other')
    imp_label = IMPERIAL_LABEL
    if imp_label is None:
        cands = [l for l in vc.index if 'inka' in l.lower() and 'late' in l.lower()]
        imp_label = cands[0] if cands else None
    idx = nonnull.index.intersection(F.index)
    R['A1'] = dict(
        total=int(len(reg)),
        has_value=int(META[REGION].notna().sum()) if REGION else 0,
        labeled=int(len(nonnull)),
        n_used=int(len(idx)),
        classes_ge10=int((vc >= 10).sum()),
        excl_other=int((vc10 != 'Other').sum()),
        imperial_label=imp_label,
    )

    # ---- A2: SHAP naming ----
    y_imp = (nonnull.loc[idx] == imp_label).astype(int).values
    Xl = F.loc[idx]
    m_a2 = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=SEED, eval_metric='logloss', n_jobs=1).fit(Xl, y_imp)
    sv = shap.TreeExplainer(m_a2).shap_values(Xl)
    sv = sv[1] if isinstance(sv, list) else sv
    imp_s = pd.Series(np.abs(sv).mean(0), index=Xl.columns).sort_values(ascending=False)
    top10 = imp_s.head(10)
    def _kind(n):
        if 'twist' in n or n == 'ratio_SZ': return 'CORD twist'
        if 'knot_dir' in n or n == 'ratio_knot_SZ': return 'KNOT direction'
        return 'other'
    R['A2'] = dict(
        top_features=list(top10.index),
        top_values=[float(x) for x in top10.values],
        kinds=[_kind(n) for n in top10.index],
        spearman_value_vs_shap={n: float(stats.spearmanr(Xl[n], sv[:, list(Xl.columns).index(n)])[0]) for n in top10.index[:5]},
    )

    # ---- A3: Unrecorded twist test (uses strict role mapping) ----
    tr_col = roles['twist_ratio'][0]
    def rank_biserial(a, b):
        u = stats.mannwhitneyu(a, b, alternative='two-sided')
        return float(u.pvalue), float(1 - 2 * u.statistic / (len(a) * len(b)))
    def boot_ci(a, b, n=2000, seed=SEED):
        rng = np.random.default_rng(seed)
        d = [np.median(rng.choice(a, len(a))) - np.median(rng.choice(b, len(b))) for _ in range(n)]
        return [float(x) for x in np.percentile(d, [2.5, 97.5])]
    A = pd.DataFrame({'cl': CL, 'twist_ratio': F[tr_col], 'pct_twist_U': DOC['pct_twist_U']})
    a = A[A.cl == COLL_CL]; b = A[A.cl == MAIN_CLUSTER]
    p_all, rb_all = rank_biserial(a.twist_ratio.dropna(), b.twist_ratio.dropna())
    ra = a[a.pct_twist_U < 0.2].twist_ratio.dropna()
    rbm = b[b.pct_twist_U < 0.2].twist_ratio.dropna()
    p_rec, rb_rec = rank_biserial(ra, rbm) if len(ra) > 5 and len(rbm) > 5 else (np.nan, np.nan)
    ci_rec = boot_ci(ra.values, rbm.values) if len(ra) > 5 and len(rbm) > 5 else [np.nan, np.nan]
    R['A3'] = dict(
        twist_col=tr_col, p_all=p_all, rb_all=rb_all, p_recorded=p_rec, rb_recorded=rb_rec,
        ci_recorded=ci_rec, n_rec=(int(len(ra)), int(len(rbm))),
        unrec_by_cluster=A.groupby('cl')['pct_twist_U'].mean().round(3).to_dict(),
    )

    # ---- B1: Size control ----
    size_cols = roles['size']
    S = np.log1p(F[size_cols].values)
    Xr = F.drop(columns=size_cols).copy()
    r2 = {}
    for c in Xr.columns:
        lr = LinearRegression().fit(S, F[c].values)
        r2[c] = float(lr.score(S, F[c].values))
        Xr[c] = F[c].values - lr.predict(S)
    Z1 = StandardScaler().fit_transform(Xr.values)
    emb1, lab1, sil1 = run_cluster(Z1)
    ok_b1 = (lab0 >= 0) & (lab1 >= 0)
    ari = float(adjusted_rand_score(lab0[ok_b1], lab1[ok_b1])) if ok_b1.sum() > 1 else np.nan
    R['B1'] = dict(
        r2_size=r2, n_clusters=int(len(set(lab1)) - (1 if -1 in lab1 else 0)),
        silhouette=float(sil1), noise_share=float(np.mean(lab1 < 0)), ari_vs_baseline=ari,
    )

    # ---- B2: Null models (optional here; full N=1000+1000 runs in B7.5) ----
    if run_nulls:
        def _report_null(tag, Z, obs_k, obs_s, bseed):
            res = {}
            for ki, kind in enumerate(['perm', 'gauss']):
                arr = run_null_suite(Z, kind, n_sim, bseed + 10000 * ki, UMAP_KW, HDB_KW, n_workers=min(6, os.cpu_count() or 1))
                cnt_s = int(np.nansum(arr[:, 1] >= obs_s))
                cnt_k = int(np.nansum(arr[:, 0] >= obs_k))
                ps = float((cnt_s + 1) / (n_sim + 1))
                pk = float((cnt_k + 1) / (n_sim + 1))
                res[kind] = dict(p_sil=ps, p_sil_fmt=format_p(cnt_s, n_sim), p_k=pk, null_sil_mean=float(np.nanmean(arr[:, 1])))
            return res
        R['B2_baseline'] = _report_null('baseline', Z0, R['baseline']['n_clusters'], sil0, 1000)
        R['B2_size_controlled'] = _report_null('size-controlled', Z1, R['B1']['n_clusters'], sil1, 50000)

    # ---- B3: Cataloger + documentation confound ----
    ok = CL >= 0
    ct_cat = pd.crosstab(CL[ok], CATALOGER[ok].fillna('NA'))
    R['B3_cramers_v_cataloger'] = cramers_v(ct_cat)
    if MUSEUM:
        top_m = META[MUSEUM].fillna('NA').value_counts().head(8).index
        ct_m = pd.crosstab(CL[ok], META[MUSEUM].fillna('NA').where(META[MUSEUM].fillna('NA').isin(top_m), 'other')[ok])
        R['B3_cramers_v_museum'] = cramers_v(ct_m)
    R['B3_doc_by_cluster'] = DOC[ok].groupby(CL[ok]).mean().round(3).to_dict()
    Xd = np.column_stack([DOC[ok].fillna(1.0).values, pd.get_dummies(CATALOGER[ok].fillna('NA')).values])
    yd = CL[ok].values
    base_maj = float(pd.Series(yd).value_counts(normalize=True).max())
    acc_doc = float(cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(Xd), yd, cv=5).mean())
    R['B3_doc_only_acc'] = dict(acc=acc_doc, majority=base_maj)

    # ---- B4: Grouped CV by cataloger ----
    def cv_f1(Xm, ym, groups=None, n=5):
        classes = np.unique(ym); enc = {c: i for i, c in enumerate(classes)}; yi = np.array([enc[c] for c in ym])
        if groups is not None:
            n = min(n, len(set(groups)))
            if n < 3: return np.nan, np.nan
            kf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=SEED)
            splits = kf.split(Xm, yi, groups)
        else:
            kf = StratifiedKFold(n_splits=n, shuffle=True, random_state=SEED)
            splits = kf.split(Xm, yi)
        pred = np.zeros(len(yi), int)
        for tr, te in splits:
            le = LabelEncoder().fit(yi[tr])
            if len(le.classes_) < 2:
                pred[te] = le.classes_[0]; continue
            mdl = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
            mdl.fit(Xm.iloc[tr], le.transform(yi[tr]))
            pred[te] = le.inverse_transform(mdl.predict(Xm.iloc[te]))
        f1w = float(f1_score(yi, pred, average='weighted'))
        f1i = float(f1_score(yi == enc[imp_label], pred == enc[imp_label])) if imp_label in enc else np.nan
        return f1w, f1i

    lab_y = vc10.loc[idx].values
    grp = CATALOGER.loc[idx].fillna('NA').values
    b4_rows = []
    for fname, Xm in [('structural only', F.loc[idx]), ('structural + documentation', F.loc[idx].join(DOC.loc[idx]))]:
        for cvname, g in [('stratified (paper)', None), ('grouped by cataloger', grp)]:
            w, i = cv_f1(Xm.fillna(0), lab_y, g)
            b4_rows.append(dict(features=fname, cv=cvname, F1_weighted=round(w, 3), F1_imperial=round(i, 3)))
    R['B4'] = b4_rows

    # ---- B5: Imperial class vs cataloger vs documentation ----
    lab_s = pd.Series(lab_y, index=idx); cat_s = CATALOGER.loc[idx].fillna('NA')
    y_all = (lab_s == imp_label).astype(int).values
    ct5 = pd.crosstab(lab_s, cat_s)
    R['B5'] = dict(
        n_imperial=int(y_all.sum()),
        imperial_by_cataloger={str(k): int(v) for k, v in ct5.loc[imp_label].items() if v > 0},
        cataloger_only_f1w=float(cv_f1(pd.get_dummies(cat_s).astype(float), lab_y)[0]),
    )
    def f1_bin(X, y):
        k = int(min(5, y.sum()))
        m = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
        return float(f1_score(y, cross_val_predict(m, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=SEED))))
    def perm_test(X, y, n=n_perm):
        obs = f1_bin(X, y)
        if not RUN_PERMS:
            return obs, float('nan')
        rng = np.random.default_rng(SEED)
        cnt = sum(f1_bin(X, rng.permutation(y)) >= obs for _ in range(n))
        return obs, float((cnt + 1) / (n + 1))

    drop_td = roles['twist_and_dir']
    geo_cols = roles['geometry']
    sets = {
        'structure (all features)':        F.loc[idx].fillna(0),
        'structure w/o twist & direction': F.loc[idx].drop(columns=drop_td).fillna(0),
        'geometry only':                   F.loc[idx][geo_cols].fillna(0),
        'documentation only':              DOC.loc[idx].fillna(1.0),
    }
    imp_tests = {}
    for sname, Xs in sets.items():
        f_val, p_val = perm_test(Xs, y_all)
        imp_tests[sname] = dict(f1=round(f_val, 3), p=round(p_val, 3))
    R['B5']['imperial_tests'] = imp_tests
    imp_ids = lab_s.index[lab_s == imp_label]; oth_ids = lab_s.index[lab_s != imp_label]
    cmp5 = pd.DataFrame({'imperial': DOC.loc[imp_ids].mean(), 'other labeled': DOC.loc[oth_ids].mean()}).round(3)
    R['B5']['unrecorded_imperial'] = cmp5['imperial'].to_dict()
    R['B5']['unrecorded_other'] = cmp5['other labeled'].to_dict()
    R['B5']['imperial_by_cluster'] = {int(k): int(v) for k, v in CL.loc[imp_ids].value_counts().items()}

    # ---- B6: Knot-type bigrams on the exact same feature set & rows ----
    ng_df = build_bigram_df(con)
    X_base = F.loc[idx].fillna(0)
    X_ng_aligned = ng_df.reindex(idx).fillna(0)
    X_plus_ng = pd.concat([X_base, X_ng_aligned], axis=1)
    w_base_s, i_base_s = cv_f1(X_base, lab_y, None)
    w_ng_s,   i_ng_s   = cv_f1(X_plus_ng, lab_y, None)
    w_base_g, i_base_g = cv_f1(X_base, lab_y, grp)
    w_ng_g,   i_ng_g   = cv_f1(X_plus_ng, lab_y, grp)
    R['B6_bigrams'] = dict(
        n_rows=int(len(idx)),
        n_with_seq=int(idx.isin(ng_df.index).sum()),
        n_bigram_features=int(ng_df.shape[1]),
        stratified=dict(base_f1w=round(w_base_s, 3), plus_ng_f1w=round(w_ng_s, 3), delta_f1w=round(w_ng_s - w_base_s, 3),
                        base_f1_imp=round(i_base_s, 3), plus_ng_f1_imp=round(i_ng_s, 3)),
        grouped=dict(base_f1w=round(w_base_g, 3), plus_ng_f1w=round(w_ng_g, 3), delta_f1w=round(w_ng_g - w_base_g, 3),
                     base_f1_imp=round(i_base_g, 3), plus_ng_f1_imp=round(i_ng_g, 3)),
    )

    artifacts = dict(F=F, DOC=DOC, META=META, CL=CL, CATALOGER=CATALOGER, emb0=emb0, lab0=lab0, ct_cat=ct_cat, top10=top10)
    return R, artifacts


## 3. Execute Sensitivity Matrix (`v1-exact` / `v1-corregida` / `robustez`)


In [5]:
con = open_db()
ALL_RESULTS = {}
PRIMARY_ART = None

modes_to_run = ['v1-exact', 'v1-corregida', 'robustez'] if RUN_MODE == 'matrix' else [RUN_MODE]
for m in modes_to_run:
    with step(f'Run {m}'):
        csv_arg = FEATURES_CSV if RUN_MODE != 'matrix' else None
        res, art = run_full_analysis(m, con, features_csv=csv_arg, run_nulls=RUN_NULLS, n_sim=N_SIM, n_perm=N_PERM, verbose=True)
        ALL_RESULTS[m] = res
        if m == (PRIMARY_MODE if RUN_MODE == 'matrix' else RUN_MODE):
            PRIMARY_ART = art

def _b4_val(R, feat, cv_prefix, col):
    for row in R.get('B4', []):
        if row['features'] == feat and row['cv'].startswith(cv_prefix):
            return row[col]
    return np.nan

sens_path = os.path.join(RESULTS_DIR, 'table_sensitivity_matrix.csv')
prev_sens = pd.read_csv(sens_path).set_index('Corrida') if (not RUN_NULLS and os.path.exists(sens_path)) else None

summary_rows = []
for m, R in ALL_RESULTS.items():
    imp_t = R['B5']['imperial_tests']
    b2b = R.get('B2_baseline', {})
    b2s = R.get('B2_size_controlled', {})
    p_g  = b2b.get('gauss', {}).get('p_sil', float(prev_sens.loc[m, 'B2_p_gauss']) if prev_sens is not None and m in prev_sens.index else np.nan)
    p_p  = b2b.get('perm', {}).get('p_sil',  float(prev_sens.loc[m, 'B2_p_perm'])  if prev_sens is not None and m in prev_sens.index else np.nan)
    ps_g = b2s.get('gauss', {}).get('p_sil', float(prev_sens.loc[m, 'B2sc_p_gauss']) if prev_sens is not None and m in prev_sens.index else np.nan)
    ps_p = b2s.get('perm', {}).get('p_sil',  float(prev_sens.loc[m, 'B2sc_p_perm'])  if prev_sens is not None and m in prev_sens.index else np.nan)
    summary_rows.append({
        'Corrida': m,
        'Features': R['n_features'],
        'n': R['n'],
        'Clusters': str(R['baseline']['sorted_sizes']),
        'Sil': round(R['baseline']['silhouette'], 3),
        'B1_ARI': round(R['B1']['ari_vs_baseline'], 2),
        'B2_p_gauss': round(p_g, 3) if not pd.isna(p_g) else np.nan,
        'B2_p_perm': round(p_p, 3) if not pd.isna(p_p) else np.nan,
        'B2sc_p_gauss': round(ps_g, 3) if not pd.isna(ps_g) else np.nan,
        'B2sc_p_perm': round(ps_p, 3) if not pd.isna(ps_p) else np.nan,
        'B3_V_cat': round(R['B3_cramers_v_cataloger'], 2),
        'B3_V_mus': round(R['B3_cramers_v_museum'], 2),
        'B3_doc_acc': round(R['B3_doc_only_acc']['acc'], 2),
        'B3_maj': round(R['B3_doc_only_acc']['majority'], 2),
        'A1_lbl': R['A1']['labeled'],
        'B4_F1w_strat': _b4_val(R, 'structural only', 'stratified', 'F1_weighted'),
        'B4_F1i_strat': _b4_val(R, 'structural only', 'stratified', 'F1_imperial'),
        'B4_F1w_grp': _b4_val(R, 'structural only', 'grouped', 'F1_weighted'),
        'B4_F1i_grp': _b4_val(R, 'structural only', 'grouped', 'F1_imperial'),
        'B5_n_imp': R['B5']['n_imperial'],
        'B5_all_F1': imp_t['structure (all features)']['f1'],
        'B5_notd_F1': imp_t['structure w/o twist & direction']['f1'],
        'B5_geo_F1': imp_t['geometry only']['f1'],
        'B5_doc_F1': imp_t['documentation only']['f1'],
        'B6_dF1_strat': R['B6_bigrams']['stratified']['delta_f1w'],
        'B6_dF1_grp': R['B6_bigrams']['grouped']['delta_f1w'],
    })

SENS_DF = pd.DataFrame(summary_rows)
SENS_DF.to_csv(sens_path, index=False)
print('\n================ SENSITIVITY MATRIX ================')
print(SENS_DF.to_string(index=False))


DB: C:\Users\maria\.gemini\antigravity\scratch\Khipu-ML\data\open-khipu-repository-2.1.0\data\khipu.db

===== Run v1-exact =====


[resolve_roles] Detected schema: v1
  twist_ratio     (1): ['ratio_SZ']
  twist_and_dir   (6): ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
  geometry        (7): ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns']
  size            (2): ['n_cords', 'n_knots']
[v1-exact] n=619, p=27, meta_cov=100.0%, doc_cov=98.9%


[v1-exact] Baseline clusters: {0: 17, 1: 159, 2: 443} | silhouette=0.817
[v1-exact] Reproduces v1 within UMAP tolerance (got [17, 159, 443], sil=0.817 vs ref [17, 160, 442], 0.769)



===== Run v1-corregida =====


[resolve_roles] Detected schema: v1
  twist_ratio     (1): ['ratio_SZ']
  twist_and_dir   (6): ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
  geometry        (7): ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns']
  size            (2): ['n_cords', 'n_knots']
[v1-corregida] n=612, p=27, meta_cov=100.0%, doc_cov=100.0%


[v1-corregida] Baseline clusters: {0: 10, 1: 160, 2: 442} | silhouette=0.793



===== Run robustez =====


[resolve_roles] Detected schema: v2
  twist_ratio     (1): ['twist_ratio']
  twist_and_dir   (5): ['n_twist_S', 'n_twist_Z', 'twist_ratio', 'knot_dir_S', 'knot_dir_Z']
  geometry        (7): ['n_cords', 'cord_len_mean', 'cord_len_std', 'max_level', 'subsidiary_ratio', 'n_knots', 'knot_turns_mean']
  size            (2): ['n_cords', 'n_knots']
[robustez] n=612, p=23, meta_cov=100.0%, doc_cov=100.0%


[robustez] Baseline clusters: {0: 12, 1: 170, 2: 430} | silhouette=0.763



================ SENSITIVITY MATRIX ================
     Corrida  Features   n       Clusters   Sil  B1_ARI  B2_p_gauss  B2_p_perm  B2sc_p_gauss  B2sc_p_perm  B3_V_cat  B3_V_mus  B3_doc_acc  B3_maj  A1_lbl  B4_F1w_strat  B4_F1i_strat  B4_F1w_grp  B4_F1i_grp  B5_n_imp  B5_all_F1  B5_notd_F1  B5_geo_F1  B5_doc_F1  B6_dF1_strat  B6_dF1_grp
    v1-exact        27 619 [17, 159, 443] 0.817    0.97        0.01      0.178          0.01        0.376      0.79      0.31        0.95    0.72     120         0.466         0.865       0.008         0.0        18      0.824       0.824      0.824      0.882         0.007       0.000
v1-corregida        27 612 [10, 160, 442] 0.793    0.90        0.01      0.109          0.01        0.158      0.73      0.30        0.95    0.72     113         0.474         0.762       0.023         0.0        11      0.778       0.778      0.778      0.800        -0.010      -0.015
    robustez        23 612 [12, 170, 430] 0.763    0.96        0.01      0.277       

---
## 4. Section B7 — `CREATED_BY = jclindaniel` Audit & Clean-Corpus Reclustering

Esta sección documenta lo observado en `cord.TWIST` por usuario que cargó el registro (`khipu_main.CREATED_BY`) y evalúa si persiste estructura de clustering al excluir secuencialmente:
1. Registros sin cordeles (`empty_no_cords`, `n = 7`)
2. Khipus con $\ge 20\%$ de torsión no registrada (`pct_twist_U_ge_0.20`, `n = 145`)
3. Khipus cargados por `jclindaniel` (`created_by_jclindaniel`, `n = 37`)

> **Nota metodológica:** los registros de `jclindaniel` se tratan como torsión no comparable (excluidos del subconjunto limpio con indicador explícito en `results/excluded_khipus_okr_2.1.0.csv`), **sin invertir ni recodificar** `S`/`Z`. La causa de la inversión S↔Z observada en Incahuasi y Armatambo se mantiene como **hipótesis pendiente; consulta enviada a OKR**.


In [6]:
# ===================== B7 CONFIG =====================
B7_SEED          = 42
B7_N_SIM         = 1000
B7_N_WORKERS     = min(6, os.cpu_count() or 1)
B7_RUN_NULLS     = False     # Set True to recompute all 1,000+1,000 nulls; False loads results/table_b2_1000.csv if present
B7_ZENODO_VER    = OKR_VERSION
B7_ZENODO_DOI    = f'10.5281/zenodo.{ZENODO_ID}'
B7_EXPECTED_META = ['okr_num', 'investigator_num', 'created_by', 'museum_name', 'provenance', 'region']
B7_EXPECTED_FEAT = ['n_cords', 'ratio_SZ', 'twist_S', 'twist_Z', 'n_knots', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
# =====================================================

F619_b7, DOC619_b7, META619_b7, _, roles_b7 = load_dataset('v1-exact', con, verbose=False)
missing_meta = [c for c in B7_EXPECTED_META if c not in META619_b7.columns]
missing_feat = [c for c in B7_EXPECTED_FEAT if c not in F619_b7.columns]
assert not missing_meta, f'B7 missing metadata columns: {missing_meta}'
assert not missing_feat, f'B7 missing feature columns: {missing_feat}'
assert 'pct_twist_U' in DOC619_b7.columns, 'B7 missing pct_twist_U in DOC'
print(f'[B7 Config] OK | OKR Zenodo {B7_ZENODO_VER} | n={len(F619_b7)} | B7_N_SIM={B7_N_SIM} | workers={B7_N_WORKERS} | B7_RUN_NULLS={B7_RUN_NULLS}')


[B7 Config] OK | OKR Zenodo 2.1.0 | n=619 | B7_N_SIM=1000 | workers=6 | B7_RUN_NULLS=False


In [7]:
with step('B7.1 jclindaniel audit by site and CREATED_BY'):
    q_sites = '''
    SELECT
        CASE
            WHEN k.PROVENANCE LIKE '%Incahuasi%' OR k.MUSEUM_NAME LIKE '%Incahu%' THEN 'Incahuasi'
            WHEN k.PROVENANCE LIKE '%Armatambo%' OR k.PROVENANCE LIKE '%Huaca San Pedro%' THEN 'Armatambo / Huaca San Pedro'
        END AS site,
        SUBSTR(k.INVESTIGATOR_NUM, 1, 2) AS cataloger_prefix,
        k.CREATED_BY,
        COUNT(DISTINCT k.KHIPU_ID) AS n_khipus,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,'')))='S' THEN 1 ELSE 0 END) AS cords_S,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,'')))='Z' THEN 1 ELSE 0 END) AS cords_Z,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,''))) NOT IN ('S','Z') THEN 1 ELSE 0 END) AS cords_U
    FROM khipu_main k
    JOIN cord c ON k.KHIPU_ID = c.KHIPU_ID
    WHERE k.PROVENANCE LIKE '%Incahuasi%' OR k.MUSEUM_NAME LIKE '%Incahu%'
       OR k.PROVENANCE LIKE '%Armatambo%' OR k.PROVENANCE LIKE '%Huaca San Pedro%'
    GROUP BY site, cataloger_prefix, k.CREATED_BY
    ORDER BY site, n_khipus DESC
    '''
    b7_sites_df = pd.read_sql(q_sites, con)
    b7_sites_df.to_csv(os.path.join(RESULTS_DIR, 'table_b7_jclindaniel_sites.csv'), index=False)
    print(b7_sites_df.to_string(index=False))
    print('\nNota: contraste S/Z observado dentro de los mismos dos sitios; causa S<->Z = hipotesis pendiente; consulta enviada a OKR.')



===== B7.1 jclindaniel audit by site and CREATED_BY =====
                       site cataloger_prefix  CREATED_BY  n_khipus  cords_S  cords_Z  cords_U
Armatambo / Huaca San Pedro               UR jclindaniel        13      117      620        1
Armatambo / Huaca San Pedro               HP       julia         1       32        0        2
                  Incahuasi               UR       julia        28     4501        0        0
                  Incahuasi               JC jclindaniel        23        1     1214        2
                  Incahuasi               UR      gurton         1       14        0        0

Nota: contraste S/Z observado dentro de los mismos dos sitios; causa S<->Z = hipotesis pendiente; consulta enviada a OKR.


In [8]:
with step('B7.2 Export excluded_khipus_okr_2.1.0.csv'):
    # Sequential filter order: 1) empty_no_cords -> 2) pct_twist_U_ge_0.20 -> 3) created_by_jclindaniel
    is_empty = F619_b7['n_cords'] == 0
    is_unrec = (~is_empty) & (DOC619_b7['pct_twist_U'] >= 0.20)
    is_jc    = (~is_empty) & (~is_unrec) & META619_b7['created_by'].fillna('').str.lower().str.contains('clindaniel')

    reason = pd.Series('', index=F619_b7.index)
    reason.loc[is_empty] = 'empty_no_cords'
    reason.loc[is_unrec] = 'pct_twist_U_ge_0.20'
    reason.loc[is_jc]    = 'created_by_jclindaniel'

    excl_full = pd.DataFrame({
        'KHIPU_ID': F619_b7.index,
        'OKR_NUM': META619_b7['okr_num'].values,
        'INVESTIGATOR_NUM': META619_b7['investigator_num'].values,
        'CREATED_BY': META619_b7['created_by'].values,
        'MUSEUM_NAME': META619_b7['museum_name'].values,
        'PROVENANCE': META619_b7['provenance'].values,
        'REGION': META619_b7['region'].values,
        'n_cords': F619_b7['n_cords'].astype(int).values,
        'pct_twist_U': DOC619_b7['pct_twist_U'].round(4).values,
        'ratio_SZ': F619_b7['ratio_SZ'].round(4).values,
        'exclusion_reason': reason.values,
        'okr_zenodo_version': B7_ZENODO_VER,
        'okr_zenodo_doi': B7_ZENODO_DOI,
    })
    excl_df = excl_full[excl_full['exclusion_reason'] != ''].copy()
    excl_path = os.path.join(RESULTS_DIR, f'excluded_khipus_okr_{B7_ZENODO_VER}.csv')
    excl_df.to_csv(excl_path, index=False)
    print(f'Total={len(F619_b7)} | Excluded={len(excl_df)} | Clean remaining={len(F619_b7) - len(excl_df)}')
    print(excl_df['exclusion_reason'].value_counts().to_string())
    print('Saved:', excl_path)



===== B7.2 Export excluded_khipus_okr_2.1.0.csv =====
Total=619 | Excluded=189 | Clean remaining=430
exclusion_reason
pct_twist_U_ge_0.20       145
created_by_jclindaniel     37
empty_no_cords              7
Saved: C:\Users\maria\.gemini\antigravity\scratch\Khipu-ML\results\excluded_khipus_okr_2.1.0.csv


In [9]:
with step('B7.3 Progression table 612 / 467 / 430'):
    F612_b7 = F619_b7.loc[~is_empty].copy()
    F467_b7 = F619_b7.loc[(~is_empty) & (~is_unrec)].copy()
    F430_b7 = F619_b7.loc[(~is_empty) & (~is_unrec) & (~is_jc)].copy()
    B7_SUBSETS = [
        ('n=612 (v1-corregida)',             F612_b7),
        ('n=467 (recorded <20% U)',          F467_b7),
        ('n=430 (recorded, no jclindaniel)', F430_b7),
    ]
    CAT619 = META619_b7['investigator_num'].astype(str).str.extract(r'^([A-Za-z]{2})')[0].str.upper().fillna('NA')
    prog_rows = []
    for name, Fsub in B7_SUBSETS:
        Z = StandardScaler().fit_transform(Fsub.values)
        _, lab, sil = run_cluster(Z, seed=B7_SEED)
        ok = lab >= 0
        k = int(len(set(lab[ok])))
        noise_pct = round(100.0 * float(np.mean(lab < 0)), 1)
        v_cat = cramers_v(pd.crosstab(pd.Series(lab, index=Fsub.index)[ok], CAT619.loc[Fsub.index][ok])) if k >= 2 else np.nan
        # Ablation without twist & knot direction (21 features)
        F_nt = Fsub.drop(columns=roles_b7['twist_and_dir'])
        _, lab_nt, sil_nt = run_cluster(StandardScaler().fit_transform(F_nt.values), seed=B7_SEED)
        k_nt = int(len(set(lab_nt[lab_nt >= 0])))
        prog_rows.append({
            'subset': name, 'n': len(Fsub),
            'k': k, 'silhouette': round(float(sil), 3), 'noise_pct': noise_pct,
            'cramers_v_cat': round(v_cat, 3) if not pd.isna(v_cat) else np.nan,
            'ratio_SZ_mean': round(float(Fsub['ratio_SZ'].mean()), 4),
            'ratio_SZ_var': round(float(Fsub['ratio_SZ'].var()), 4),
            'no_td21_k': k_nt, 'no_td21_sil': round(float(sil_nt), 3),
            'no_td21_noise_pct': round(100.0 * float(np.mean(lab_nt < 0)), 1),
        })
    B7_PROG_DF = pd.DataFrame(prog_rows)
    B7_PROG_DF.to_csv(os.path.join(RESULTS_DIR, 'table_b7_progression.csv'), index=False)
    print(B7_PROG_DF.to_string(index=False))



===== B7.3 Progression table 612 / 467 / 430 =====


                          subset   n  k  silhouette  noise_pct  cramers_v_cat  ratio_SZ_mean  ratio_SZ_var  no_td21_k  no_td21_sil  no_td21_noise_pct
            n=612 (v1-corregida) 612  3       0.793        0.0          0.733         0.6946        0.2087          2        0.807                0.0
         n=467 (recorded <20% U) 467  2       0.591        0.0          0.756         0.8942        0.0909         11        0.446               29.3
n=430 (recorded, no jclindaniel) 430  3       0.210        3.3          0.174         0.9700        0.0258         11        0.381               17.7


In [10]:
with step('B7.4 Hyperparameter sweep (18 configs per subset)'):
    sweep_rows = []
    for name, Fsub in B7_SUBSETS:
        Z = StandardScaler().fit_transform(Fsub.values)
        for nn in [10, 15, 30]:
            for md in [0.0, 0.1]:
                emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md, metric='euclidean', random_state=B7_SEED).fit_transform(Z)
                for mcs in [10, 15, 25]:
                    lab = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=5, metric='euclidean').fit_predict(emb)
                    ok = lab >= 0
                    k = int(len(set(lab[ok])))
                    sil = float(silhouette_score(emb[ok], lab[ok])) if ok.sum() > 1 and k > 1 else np.nan
                    sweep_rows.append(dict(
                        subset=name, n=len(Fsub), n_neighbors=nn, min_dist=md, min_cluster_size=mcs,
                        k=k, sil=round(sil, 3) if not np.isnan(sil) else np.nan,
                        noise_pct=round(100.0 * float(np.mean(lab < 0)), 1),
                    ))
    B7_SWEEP_DF = pd.DataFrame(sweep_rows)
    B7_SWEEP_DF.to_csv(os.path.join(RESULTS_DIR, 'table_hyperparam_sweep.csv'), index=False)
    print(B7_SWEEP_DF.groupby('subset').agg(
        configs=('k', 'count'),
        k_min=('k', 'min'), k_med=('k', 'median'), k_max=('k', 'max'),
        sil_min=('sil', 'min'), sil_med=('sil', 'median'), sil_max=('sil', 'max'),
        noise_med=('noise_pct', 'median'),
    ).round(3).to_string())



===== B7.4 Hyperparameter sweep (18 configs per subset) =====


                                  configs  k_min  k_med  k_max  sil_min  sil_med  sil_max  noise_med
subset                                                                                              
n=430 (recorded, no jclindaniel)       18      2    3.0     13    0.210    0.380    0.458       11.4
n=467 (recorded <20% U)                18      2    2.0      2    0.529    0.619    0.714        0.0
n=612 (v1-corregida)                   18      2    3.0      7    0.680    0.756    0.819        1.6


In [11]:
with step('B7.5 B2 null models (N_SIM=1000+1000)'):
    b2_path = os.path.join(RESULTS_DIR, 'table_b2_1000.csv')
    if (not B7_RUN_NULLS) and os.path.exists(b2_path):
        B7_NULL_DF = pd.read_csv(b2_path)
        print(f'Loaded cached B2 results from {b2_path} (set B7_RUN_NULLS=True to recompute):')
        print(B7_NULL_DF.to_string(index=False))
    else:
        b2_rows = []
        for s_idx, (name, Fsub) in enumerate(B7_SUBSETS):
            Z = StandardScaler().fit_transform(Fsub.values)
            _, lab_obs, sil_obs = run_cluster(Z, seed=B7_SEED)
            k_obs = int(len(set(lab_obs[lab_obs >= 0])))
            row = {'subset': name, 'n': len(Fsub), 'k_obs': k_obs, 'sil_obs': round(float(sil_obs), 3)}
            for k_idx, kind in enumerate(['perm', 'gauss']):
                base_s = 100000 * (s_idx + 1) + 10000 * (k_idx + 1)
                arr = run_null_suite(Z, kind, B7_N_SIM, base_s, UMAP_KW, HDB_KW, n_workers=B7_N_WORKERS)
                cnt_s = int(np.nansum(arr[:, 1] >= sil_obs))
                cnt_k = int(np.nansum(arr[:, 0] >= k_obs))
                row[f'{kind}_null_sil_mean'] = round(float(np.nanmean(arr[:, 1])), 3)
                row[f'{kind}_null_sil_sd']   = round(float(np.nanstd(arr[:, 1])), 3)
                row[f'{kind}_null_sil_p95']  = round(float(np.nanpercentile(arr[:, 1], 95)), 3)
                row[f'{kind}_count_ge']      = cnt_s
                row[f'{kind}_p_sil']         = format_p(cnt_s, B7_N_SIM)
                row[f'{kind}_p_k']           = format_p(cnt_k, B7_N_SIM)
            b2_rows.append(row)
        B7_NULL_DF = pd.DataFrame(b2_rows)
        B7_NULL_DF.to_csv(b2_path, index=False)
        print(B7_NULL_DF.to_string(index=False))



===== B7.5 B2 null models (N_SIM=1000+1000) =====
Loaded cached B2 results from C:\Users\maria\.gemini\antigravity\scratch\Khipu-ML\results\table_b2_1000.csv (set B7_RUN_NULLS=True to recompute):
                          subset   n  k_obs  sil_obs  perm_null_sil_mean  perm_null_sil_sd  perm_null_sil_p95  perm_p_sil  perm_p_k  gauss_null_sil_mean  gauss_null_sil_sd  gauss_null_sil_p95 gauss_p_sil  gauss_p_k
n=430 (recorded, no jclindaniel) 430      3    0.210               0.492             0.187              0.755      0.9151    0.1648                0.336              0.123               0.480      0.8511     0.8611
         n=467 (recorded <20% U) 467      2    0.591               0.408             0.167              0.623      0.1029    1.0000                0.324              0.136               0.471  p <= 0.001     1.0000
            n=612 (v1-corregida) 612      3    0.793               0.538             0.211              0.818      0.1069    0.3427                0.301      

## 5. Figures & JSON Export (Primary Mode)


In [12]:
pm = PRIMARY_MODE if RUN_MODE == 'matrix' else RUN_MODE
R = ALL_RESULTS[pm]

with step('Figures'):
    emb0 = PRIMARY_ART['emb0']; lab0 = PRIMARY_ART['lab0']
    DOC = PRIMARY_ART['DOC']; CATALOGER = PRIMARY_ART['CATALOGER']
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    ax[0].scatter(emb0[:, 0], emb0[:, 1], c=lab0, cmap='tab10', s=9)
    ax[0].set_title(f'HDBSCAN clusters ({pm}, n={R["n"]})')
    sc = ax[1].scatter(emb0[:, 0], emb0[:, 1], c=DOC['pct_twist_U'].values, cmap='viridis', s=9)
    plt.colorbar(sc, ax=ax[1], label='share of cords with unrecorded twist')
    ax[1].set_title('Same embedding: unrecorded twist')
    codes = pd.Categorical(CATALOGER.fillna('NA'))
    ax[2].scatter(emb0[:, 0], emb0[:, 1], c=codes.codes, cmap='tab10', s=9)
    for i, cname in enumerate(codes.categories):
        ax[2].scatter([], [], color=plt.cm.tab10(i / 10), label=cname)
    ax[2].legend(title='cataloger', fontsize=8)
    ax[2].set_title('Same embedding: cataloger')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_umap_clusters_vs_documentation.png'), dpi=200)
    plt.close()

    a2 = R['A2']
    names = a2['top_features'][::-1]; kinds = a2['kinds'][::-1]; vals = a2['top_values'][::-1]
    colors = {'KNOT direction': '#d95f02', 'CORD twist': '#1b9e77', 'other': '#7570b3'}
    plt.figure(figsize=(7, 4.5))
    plt.barh(names, vals, color=[colors[k] for k in kinds])
    for cn, col in colors.items():
        plt.barh([], [], color=col, label=cn)
    plt.legend(); plt.xlabel('mean |SHAP| (Inka Late Horizon vs rest)')
    plt.title(f'Top features, named by what they measure ({pm})')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_shap_imperial_named.png'), dpi=200)
    plt.close()

    imp_t = R['B5']['imperial_tests']
    ks = list(imp_t); fs = [imp_t[k]['f1'] for k in ks]
    plt.figure(figsize=(8, 4))
    plt.barh(ks[::-1], fs[::-1], color='#4c72b0')
    for i, k in enumerate(ks[::-1]):
        p_lbl = f"p={imp_t[k]['p']:.3f}" if not pd.isna(imp_t[k]['p']) else f"F1={imp_t[k]['f1']:.3f}"
        plt.text(imp_t[k]['f1'] + 0.01, i, p_lbl, va='center', fontsize=9)
    plt.xlim(0, 1.08); plt.xlabel('F1 (imperial class, 5-fold CV)')
    plt.title(f"Imperial class (n={R['B5']['n_imperial']}, {pm}): what predicts it?")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_imperial_predictors.png'), dpi=200)
    plt.close()

with step('Export full JSON'):
    with open(os.path.join(OUTPUT_DIR, 'okr_v2_results.json'), 'w', encoding='utf-8') as f:
        json.dump(ALL_RESULTS, f, indent=2, default=str)

print('\n================ STEP STATUS ================')
for k, s in STATUS.items():
    print(f'{k:45s} {s}')
assert all(v == 'OK' for v in STATUS.values()), f'Some steps failed: {STATUS}'



===== Figures =====



===== Export full JSON =====

================ STEP STATUS ================
Run v1-exact                                  OK
Run v1-corregida                              OK
Run robustez                                  OK
B7.1 jclindaniel audit by site and CREATED_BY OK
B7.2 Export excluded_khipus_okr_2.1.0.csv     OK
B7.3 Progression table 612 / 467 / 430        OK
B7.4 Hyperparameter sweep (18 configs per subset) OK
B7.5 B2 null models (N_SIM=1000+1000)         OK
Figures                                       OK
Export full JSON                              OK
